# Power Spectrum, Wavepacket Features & Fourier Filtering

This notebook demonstrates:

1. A **power-law** spectrum $P(k) = A\,k^{-\alpha}$ with tuneable normalization and index.
2. A **wavepacket feature** (Gaussian-enveloped sinusoid) injected onto $P(k)$.
3. **Numerical Fourier transform** of the composite signal from $k$-space to the conjugate $x$-space.
4. A **band-pass / band-stop filter** applied in $x$-space, followed by an **inverse FFT** back to $k$-space.
5. All plots on **log–log** axes for clarity across decades.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import windows

# Plotting defaults
plt.rcParams.update({
    'font.size': 14,
    'font.family': 'STIXGeneral',
    'mathtext.fontset': 'cm',
    'figure.figsize': (12, 6),
})

---
## 1. Power-Law Spectrum $P(k)$

We define

$$
P(k) = A \, k^{-\alpha}
$$

with:
- $A$ — normalization (sets the amplitude at $k=1$),
- $\alpha$ — spectral index ($\ge 1$).

The $k$-grid is **uniformly spaced** (required for FFT later),
but plotted on log–log axes.

In [ ]:
# ============================================
# PARAMETERS — tune these freely
# ============================================
k_min   = 1e-3          # lower bound of k-grid
k_max   = 1e0           # upper bound of k-grid
N       = 2**16         # number of uniformly-spaced k samples (power of 2 for FFT)
alpha   = 1.2           # power-law spectral index
A_norm  = 2.5e-9        # normalization: P(k=1) = A_norm

# ============================================
# BUILD K-GRID AND POWER LAW
# ============================================
dk = (k_max - k_min) / N
k  = np.linspace(k_min, k_max, N, endpoint=False)

P_powerlaw = A_norm * k**(-alpha)

print(f"k range:  [{k.min():.2e}, {k.max():.2e}]")
print(f"dk:       {dk:.2e}")
print(f"P range:  [{P_powerlaw.min():.2e}, {P_powerlaw.max():.2e}]")

# ============================================
# PLOT
# ============================================
fig, ax = plt.subplots()
ax.loglog(k, P_powerlaw, 'b-', lw=2, label=rf'$P(k) = {A_norm:.1e}\;k^{{-{alpha}}}$')
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$P(k)$')
ax.set_title('Power-Law Spectrum')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Wavepacket Feature on $P(k)$ — Log-Space Construction

To produce a wavepacket that is **symmetric in log–log** space, we use an
**exponential multiplicative** perturbation with envelope and carrier
both defined in $\log_{10} k$:

$$
P_{\rm tot}(k) = P(k)\;\cdot\;10^{\;A_{\rm dex}\;\exp\!\left(-\frac{(\log_{10} k - \log_{10} k_0)^2}{2\,\sigma_{\log}^2}\right)\;\sin\bigl(2\pi\,\nu_{\log}\,(\log_{10} k - \log_{10} k_0)\bigr)}
$$

- $k_0$ — centre of the wavepacket,
- $\sigma_{\log}$ — Gaussian width in decades of $k$,
- $\nu_{\log}$ — carrier frequency (oscillations per decade),
- $A_{\rm dex}$ — peak amplitude in decades (e.g.\ 0.5 = half-decade swing).

This form guarantees:
- Peaks and troughs are **symmetric** in $\log P$ (same factor above & below $P(k)$),
- $P_{\rm tot}(k) > 0$ always (no sign flips → no log artifacts),
- Oscillation spacing is uniform on a log-$k$ axis.

In [ ]:
# ============================================
# WAVEPACKET PARAMETERS (all in log10-space)
# ============================================
k0          = 0.15       # wavepacket centre in k-space
sigma_log   = 0.06       # Gaussian envelope width in decades of k
nu_log      = 20.0       # carrier frequency (oscillations per decade)
A_dex       = 0.5        # amplitude in decades (0.5 → ×3.16 peak, ÷3.16 trough)

# ============================================
# BUILD WAVEPACKET IN LOG-SPACE (exponential form)
# ============================================
logk   = np.log10(k)
logk0  = np.log10(k0)

envelope = np.exp(-0.5 * ((logk - logk0) / sigma_log)**2)
carrier  = np.sin(2 * np.pi * nu_log * (logk - logk0))

# Exponential multiplicative perturbation → symmetric in log-log, always > 0
exponent = A_dex * envelope * carrier
P_total  = P_powerlaw * 10**exponent
W        = P_total - P_powerlaw    # the additive "excess" for downstream use

# ============================================
# PLOT
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: full composite on log-log
ax = axes[0]
ax.loglog(k, P_powerlaw, 'b-', lw=1.5, alpha=0.5, label=r'Power law $P(k)$')
ax.loglog(k, P_total, 'r-', lw=1.5, label=r'$P_{\rm tot}(k)$')
# Shade ±3σ region in log-space
k_lo = 10**(logk0 - 3*sigma_log)
k_hi = 10**(logk0 + 3*sigma_log)
ax.axvspan(k_lo, k_hi, color='gold', alpha=0.15,
           label=rf'Wavepacket region ($k_0={k0}$)')
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$P(k)$')
ax.set_title('Composite Spectrum (full range)')
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)

# Right: zoom into the wavepacket region (log-log)
ax = axes[1]
mask = (logk > logk0 - 4*sigma_log) & (logk < logk0 + 4*sigma_log)
ax.loglog(k[mask], P_total[mask], 'r-', lw=1.5, label=r'$P_{\rm tot}(k)$')
ax.loglog(k[mask], P_powerlaw[mask], 'b--', lw=1.5, alpha=0.7, label=r'Power law only')
# Upper/lower envelope (symmetric in log space)
ax.loglog(k[mask], P_powerlaw[mask] * 10**(A_dex * envelope[mask]),
          'gold', lw=1.2, alpha=0.7)
ax.loglog(k[mask], P_powerlaw[mask] * 10**(-A_dex * envelope[mask]),
          'gold', lw=1.2, alpha=0.7, label='Envelope')
ax.fill_between(k[mask],
                P_powerlaw[mask] * 10**(-A_dex * envelope[mask]),
                P_powerlaw[mask] * 10**(A_dex * envelope[mask]),
                color='gold', alpha=0.15)
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$P(k)$')
ax.set_title(f'Zoom: wavepacket near $k_0 = {k0}$ (log-log)')
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 3. Numerical Fourier Transform

We take the **discrete Fourier transform** of $P_{\rm tot}(k)$, treating $k$ as the
"time-like" coordinate.  The conjugate variable is $x$ (units of $1/k$).

A Hann window is applied before the FFT to reduce spectral leakage from the
finite $k$-range and the divergence of the power law at low $k$.

$$
\tilde{P}(x) = \mathrm{FFT}\bigl[w(k)\,P_{\rm tot}(k)\bigr]
$$

In [ ]:
# ============================================
# WINDOWING
# ============================================
window = windows.hann(N)
P_windowed = P_total * window

# ============================================
# FFT  (k-space → x-space)
# ============================================
P_fft = np.fft.fft(P_windowed)
x     = np.fft.fftfreq(N, d=dk)          # conjugate variable

# Take positive-frequency half only
pos   = x > 0
x_pos = x[pos]
amp   = np.abs(P_fft[pos])

print(f"x (conjugate) range:  [{x_pos.min():.2e}, {x_pos.max():.2e}]")
print(f"|FFT| dynamic range:  [{amp.min():.2e}, {amp.max():.2e}]")

# ============================================
# ALSO FFT THE PURE POWER LAW FOR COMPARISON
# ============================================
P_fft_pl = np.fft.fft(P_powerlaw * window)
amp_pl   = np.abs(P_fft_pl[pos])

# ============================================
# Identify the wavepacket peak in Fourier space
# (the log-space carrier is a chirp in linear k,
#  so we find the peak empirically)
# ============================================
excess = amp - amp_pl
peak_idx = np.argmax(excess)
x_peak   = x_pos[peak_idx]
print(f"Wavepacket peak in Fourier space at x = {x_peak:.1f}")

# ============================================
# PLOT
# ============================================
fig, ax = plt.subplots()
ax.loglog(x_pos, amp_pl, 'b-', lw=1.5, alpha=0.5, label=r'FFT of power law only')
ax.loglog(x_pos, amp, 'r-', lw=1.5, label=r'FFT of $P_{\rm tot}(k)$')

# Mark the empirical peak
ax.axvline(x_peak, color='green', ls='--', lw=1.5, alpha=0.7,
           label=rf'Wavepacket peak ($x={x_peak:.0f}$)')
ax.set_xlabel(r'$x$ (conjugate of $k$)')
ax.set_ylabel(r'$|\tilde{P}(x)|$')
ax.set_title('Fourier Transform of the Composite Spectrum')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

---
## 4. Filtering in Fourier Space & Inverse FFT

We apply a **band-stop (notch) filter** around the carrier frequency $\nu_k$
of the injected wavepacket, then inverse-FFT back to $k$-space.

The filter is a smooth (raised-cosine) notch that zeroes out a band
$[\nu_k - \Delta\nu,\; \nu_k + \Delta\nu]$ in $x$-space.  This should
remove the oscillatory wavepacket feature while preserving the underlying
power law.

A **band-pass** filter is also demonstrated: it *keeps* only the
wavepacket modes and discards everything else.

In [ ]:
# ============================================
# FILTER PARAMETERS
# ============================================
# Option 1: symmetric notch (set left/right to None)
# Option 2: custom asymmetric edges (set explicit values)
notch_centre    = x_peak             # centre of the notch (from empirical peak)
notch_left      = 400               # left edge of notch  (None → use centre - halfwidth)
notch_right     = 30000               # right edge of notch (None → use centre + halfwidth)
notch_halfwidth = 30.0               # symmetric half-width (used when left/right are None)
taper_width     = 10.0               # raised-cosine roll-off width (applied to both edges)

# ============================================
# BUILD BAND-STOP FILTER (notch) — supports custom edges
# ============================================
def build_notch_filter(x_arr, centre, halfwidth, taper,
                       left_edge=None, right_edge=None):
    """
    Smooth band-stop filter: 0 inside the notch, 1 outside,
    with a raised-cosine transition of width `taper`.
    Applied symmetrically to positive and negative frequencies.

    Parameters
    ----------
    x_arr : array — Fourier-space coordinate
    centre : float — notch centre frequency
    halfwidth : float — symmetric half-width (fallback)
    taper : float — raised-cosine roll-off width
    left_edge : float or None — custom left edge of the notch
                (if None, uses centre - halfwidth)
    right_edge : float or None — custom right edge of the notch
                 (if None, uses centre + halfwidth)
    """
    lo = left_edge  if left_edge  is not None else centre - halfwidth
    hi = right_edge if right_edge is not None else centre + halfwidth

    filt = np.ones_like(x_arr, dtype=float)
    xabs = np.abs(x_arr)           # work with |x| for ± symmetry

    # Core: hard zero between edges
    core = (xabs >= lo) & (xabs <= hi)
    filt[core] = 0.0

    # Left taper: roll from 1→0 as |x| goes from (lo-taper) to lo
    left_trans = (xabs >= lo - taper) & (xabs < lo)
    filt[left_trans] = 0.5 * (1 - np.cos(np.pi * (lo - xabs[left_trans]) / taper))

    # Right taper: roll from 0→1 as |x| goes from hi to (hi+taper)
    right_trans = (xabs > hi) & (xabs <= hi + taper)
    filt[right_trans] = 0.5 * (1 - np.cos(np.pi * (xabs[right_trans] - hi) / taper))

    return filt

H_notch = build_notch_filter(x, notch_centre, notch_halfwidth, taper_width,
                             left_edge=notch_left, right_edge=notch_right)

# Band-pass = complement of the notch
H_bandpass = 1.0 - H_notch

# Effective edges for labelling
eff_lo = notch_left  if notch_left  is not None else notch_centre - notch_halfwidth
eff_hi = notch_right if notch_right is not None else notch_centre + notch_halfwidth

# ============================================
# APPLY FILTERS IN FOURIER SPACE
# ============================================
P_fft_notched   = P_fft * H_notch       # remove the wavepacket
P_fft_bandpass  = P_fft * H_bandpass     # keep only the wavepacket

# ============================================
# INVERSE FFT BACK TO K-SPACE
# ============================================
P_filtered_notch    = np.fft.ifft(P_fft_notched).real / window.clip(1e-10)
P_filtered_bandpass = np.fft.ifft(P_fft_bandpass).real / window.clip(1e-10)

# Mask out edges where the window → 0 (division unreliable)
edge = int(0.02 * N)
safe = slice(edge, N - edge)

# ============================================
# PLOT: FILTER SHAPE IN X-SPACE
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.semilogx(x_pos, H_notch[pos], 'b-', lw=2, label='Band-stop (notch)')
ax.semilogx(x_pos, H_bandpass[pos], 'r--', lw=2, label='Band-pass')
ax.axvline(x_peak, color='green', ls=':', lw=1.5, alpha=0.7,
           label=rf'Peak $x={x_peak:.0f}$')
ax.axvspan(eff_lo, eff_hi, color='blue', alpha=0.08,
           label=rf'Notch [{eff_lo:.0f}, {eff_hi:.0f}]')
ax.set_xlabel(r'$x$ (conjugate of $k$)')
ax.set_ylabel('Filter gain')
ax.set_title('Filter Transfer Functions')
ax.set_ylim(-0.05, 1.1)
ax.legend()
ax.grid(True, which='both', alpha=0.3)

# Plot filtered FFT amplitudes
ax = axes[1]
ax.loglog(x_pos, amp, 'gray', lw=1, alpha=0.5, label='Unfiltered')
ax.loglog(x_pos, np.abs(P_fft_notched[pos]), 'b-', lw=1.5, label='After notch')
ax.loglog(x_pos, np.abs(P_fft_bandpass[pos]) + 1e-30, 'r-', lw=1.5, label='After band-pass')
ax.axvspan(eff_lo, eff_hi, color='blue', alpha=0.08)
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$|\tilde{P}(x)|$')
ax.set_title('FFT Amplitudes After Filtering')
ax.legend()
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# PLOT: FILTERED SPECTRA BACK IN K-SPACE
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel 1: Notch-filtered (wavepacket removed) ---
ax = axes[0]
ax.loglog(k[safe], np.abs(P_total[safe]), 'r-', lw=1, alpha=0.4,
          label=r'Original $P(k)+W(k)$')
ax.loglog(k[safe], np.abs(P_filtered_notch[safe]), 'b-', lw=2,
          label='After notch filter')
ax.loglog(k[safe], P_powerlaw[safe], 'k--', lw=1.5, alpha=0.6,
          label='True power law')
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$P(k)$')
ax.set_title('Notch Filter: Wavepacket Removed')
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)

# --- Panel 2: Band-passed (only the wavepacket) ---
ax = axes[1]
ax.loglog(k[safe], np.abs(P_total[safe]), 'r-', lw=1, alpha=0.4,
          label=r'Original $P(k)+W(k)$')
ax.loglog(k[safe], np.abs(P_filtered_bandpass[safe]) + 1e-30, 'g-', lw=2,
          label='After band-pass (wavepacket only)')
ax.loglog(k[safe], np.abs(W[safe]) + 1e-30, 'k--', lw=1.5, alpha=0.6,
          label='True wavepacket $W(k)$')
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$P(k)$')
ax.set_title('Band-Pass Filter: Wavepacket Isolated')
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Residual Analysis

How well did the notch filter recover the underlying power law?
We plot the fractional residual:

$$
\delta(k) = \frac{P_{\rm filtered}(k) - P_{\rm powerlaw}(k)}{P_{\rm powerlaw}(k)}
$$

In [ ]:
# ============================================
# FRACTIONAL RESIDUAL
# ============================================
residual_before = (P_total[safe] - P_powerlaw[safe]) / P_powerlaw[safe]
residual_after  = (P_filtered_notch[safe] - P_powerlaw[safe]) / P_powerlaw[safe]

fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogx(k[safe], residual_before, 'r-', lw=1, alpha=0.5,
            label='Before filter (with wavepacket)')
ax.semilogx(k[safe], residual_after, 'b-', lw=1.5,
            label='After notch filter')
ax.axhline(0, color='k', ls='--', lw=1)
k_lo = 10**(logk0 - 3*sigma_log)
k_hi = 10**(logk0 + 3*sigma_log)
ax.axvspan(k_lo, k_hi, color='gold', alpha=0.15,
           label='Wavepacket region')
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$(P_{\rm filt} - P_{\rm true}) / P_{\rm true}$')
ax.set_title('Fractional Residual After Notch Filtering')
ax.legend()
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

rms_before = np.std(residual_before)
rms_after  = np.std(residual_after)
print(f"RMS fractional residual BEFORE filter: {rms_before:.4e}")
print(f"RMS fractional residual AFTER  filter: {rms_after:.4e}")
print(f"Improvement factor: {rms_before/rms_after:.1f}x")

---
## 6. Noisy Wavepacket + Notch Filtering

We add **multiplicative Gaussian noise** at all $k$ scales to the composite
wavepacket spectrum $P_{\rm tot}(k)$, then repeat the full FFT → notch-filter
→ IFFT pipeline to test robustness.

The noise is applied in log-space (as a Gaussian perturbation in dex) so that
it appears symmetric on a log–log plot:

$$
P_{\rm noisy}(k) = P_{\rm tot}(k)\;\cdot\;10^{\;\mathcal{N}(0,\,\sigma_{\rm noise})}
$$

Custom left/right notch edges are used to demonstrate the asymmetric filter.

In [ ]:
# ============================================
# NOISE PARAMETERS
# ============================================
sigma_noise = 0.2       # noise amplitude in dex (0.05 → ~12% scatter)
rng_seed    = 42         # random seed for reproducibility

# NOTCH FILTER EDGES (custom left/right for noisy case)
noisy_notch_left  = 150.0    # left edge of notch in x-space
noisy_notch_right = 30000.0   # right edge of notch in x-space
noisy_taper       = 5.0     # taper width

# ============================================
# ADD GAUSSIAN NOISE IN LOG-SPACE
# ============================================
rng   = np.random.default_rng(rng_seed)
noise_dex = rng.normal(0, sigma_noise, size=N)
P_noisy   = P_total * 10**noise_dex

# ============================================
# PLOT: NOISY SPECTRUM
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.loglog(k, P_total, 'r-', lw=1, alpha=0.4, label=r'$P_{\rm tot}(k)$ (clean)')
ax.loglog(k, P_noisy, 'gray', lw=0.5, alpha=0.7, label=r'$P_{\rm noisy}(k)$')
ax.loglog(k, P_powerlaw, 'b--', lw=1.5, alpha=0.5, label=r'Power law')
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$P(k)$')
ax.set_title(f'Noisy Composite Spectrum ($\\sigma_{{\\rm noise}} = {sigma_noise}$ dex)')
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)

# Zoom
ax = axes[1]
ax.loglog(k[mask], P_noisy[mask], 'gray', lw=0.8, alpha=0.7, label=r'$P_{\rm noisy}$')
ax.loglog(k[mask], P_total[mask], 'r-', lw=1.5, label=r'$P_{\rm tot}$ (clean)')
ax.loglog(k[mask], P_powerlaw[mask], 'b--', lw=1.5, alpha=0.5, label='Power law')
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$P(k)$')
ax.set_title(f'Zoom: noisy wavepacket near $k_0 = {k0}$')
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================
# FFT OF NOISY SIGNAL
# ============================================
P_noisy_windowed = P_noisy * window
P_noisy_fft      = np.fft.fft(P_noisy_windowed)
amp_noisy        = np.abs(P_noisy_fft[pos])

# ============================================
# BUILD NOTCH FILTER WITH CUSTOM EDGES
# ============================================
H_notch_noisy = build_notch_filter(x, notch_centre, notch_halfwidth, noisy_taper,
                                   left_edge=noisy_notch_left,
                                   right_edge=noisy_notch_right)

# Apply filter + IFFT
P_noisy_fft_notched = P_noisy_fft * H_notch_noisy
P_noisy_filtered    = np.fft.ifft(P_noisy_fft_notched).real / window.clip(1e-10)

# ============================================
# PLOT: FFT + FILTER
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
ax.loglog(x_pos, amp_pl, 'b-', lw=1, alpha=0.3, label='FFT power law only')
ax.loglog(x_pos, amp_noisy, 'gray', lw=0.8, alpha=0.6, label='FFT noisy signal')
ax.loglog(x_pos, amp, 'r-', lw=1, alpha=0.4, label='FFT clean wavepacket')
ax.axvspan(noisy_notch_left, noisy_notch_right, color='blue', alpha=0.08,
           label=rf'Notch [{noisy_notch_left:.0f}, {noisy_notch_right:.0f}]')
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$|\tilde{P}(x)|$')
ax.set_title('FFT of Noisy Signal')
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)

ax = axes[1]
ax.loglog(x_pos, amp_noisy, 'gray', lw=0.8, alpha=0.5, label='Before filter')
ax.loglog(x_pos, np.abs(P_noisy_fft_notched[pos]), 'b-', lw=1.5, label='After notch')
ax.axvspan(noisy_notch_left, noisy_notch_right, color='blue', alpha=0.08)
ax.set_xlabel(r'$x$')
ax.set_ylabel(r'$|\tilde{P}(x)|$')
ax.set_title('Noisy FFT After Notch Filter')
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================
# PLOT: FILTERED NOISY SPECTRUM IN K-SPACE
# ============================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
ax.loglog(k, P_noisy, 'gray', lw=0.5, alpha=0.5,
          label=r'$P_{\rm noisy}(k)$')
ax.loglog(k, np.abs(P_noisy_filtered), 'b-', lw=1.5,
          label='After notch filter')
ax.loglog(k, P_powerlaw, 'k--', lw=1.5, alpha=0.5,
          label='True power law')
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$P(k)$')
ax.set_ylim(1e-10, 1e-5)
ax.set_title('Noisy Signal: Notch Filter Result (full range)')
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)

# Zoom on wavepacket region
ax = axes[1]
ax.loglog(k[mask], P_noisy[mask], 'gray', lw=0.8, alpha=0.5,
          label=r'$P_{\rm noisy}$')
ax.loglog(k[mask], np.abs(P_noisy_filtered[mask]), 'b-', lw=1.5,
          label='After notch filter')
ax.loglog(k[mask], P_powerlaw[mask], 'k--', lw=1.5, alpha=0.5,
          label='True power law')
ax.loglog(k[mask], P_total[mask], 'r-', lw=1, alpha=0.4,
          label=r'$P_{\rm tot}$ (clean)')
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'$P(k)$')
ax.set_title(f'Zoom: notch-filtered near $k_0 = {k0}$')
ax.legend(fontsize=11)
ax.grid(True, which='both', alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================
# RESIDUAL: NOISY FILTERED vs POWER LAW
# ============================================
res_noisy_before = (P_noisy[safe] - P_powerlaw[safe]) / P_powerlaw[safe]
res_noisy_after  = (P_noisy_filtered[safe] - P_powerlaw[safe]) / P_powerlaw[safe]

fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogx(k[safe], res_noisy_before, 'gray', lw=0.5, alpha=0.5,
            label='Noisy (before filter)')
ax.semilogx(k[safe], res_noisy_after, 'b-', lw=1,
            label='Noisy (after notch filter)')
ax.semilogx(k[safe], residual_after, 'r--', lw=1, alpha=0.5,
            label='Clean (after notch, from §5)')
ax.axhline(0, color='k', ls='--', lw=1)
k_lo = 10**(logk0 - 3*sigma_log)
k_hi = 10**(logk0 + 3*sigma_log)
ax.axvspan(k_lo, k_hi, color='gold', alpha=0.15, label='Wavepacket region')
ax.set_xlabel(r'$k$')
ax.set_ylabel(r'Fractional residual')
ax.set_title('Residual: Noisy Filtered vs True Power Law')
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

rms_noisy_before = np.std(res_noisy_before)
rms_noisy_after  = np.std(res_noisy_after)
print(f"RMS fractional residual (noisy) BEFORE filter: {rms_noisy_before:.4e}")
print(f"RMS fractional residual (noisy) AFTER  filter: {rms_noisy_after:.4e}")
print(f"Improvement factor: {rms_noisy_before/rms_noisy_after:.1f}x")

---
## Summary

| Step | What | Key parameter(s) |
|------|------|-------------------|
| 1 | Power-law spectrum | $A$, $\alpha$ |
| 2 | Inject wavepacket (log-space, exponential) | $k_0$, $\sigma_{\log}$, $\nu_{\log}$, $A_{\rm dex}$ |
| 3 | FFT $k \to x$ | Hann window, $N$ |
| 4a | **Notch filter** → remove wavepacket | `notch_centre`, `notch_halfwidth` |
| 4b | **Band-pass** → isolate wavepacket | (complement of notch) |
| 5 | Residual check | RMS fractional error |

All parameters are collected at the top of each cell for easy tuning.